# Simple MLP for text classification in PyTorch

__Data__

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

# Set maximum number of features
max_features = 1000
# Choose two categories for classification (for all available categories, see
# https://scikit-learn.org/stable/datasets/real_world.html#usage)
categories = ["sci.space", "talk.politics.misc"]

train_raw = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "footers", "quotes")
)
test_raw = fetch_20newsgroups(
    subset="test", categories=categories, remove=("headers", "footers", "quotes")
)

vectorizer = TfidfVectorizer(max_features=max_features)
X_train = vectorizer.fit_transform(train_raw.data).toarray()
X_test = vectorizer.transform(test_raw.data).toarray()

y_train = train_raw.target.reshape(-1, 1).astype(float)
y_test = test_raw.target.reshape(-1, 1).astype(float)


def to_tensors(X, y):
    return TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y))  # Float for BCE


train_loader = DataLoader(
    to_tensors(X_train, y_train.ravel()), batch_size=64, shuffle=True
)
test_loader = DataLoader(to_tensors(X_test, y_test.ravel()), batch_size=64)

__Model__

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Sigmoid(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)  # (batch,)


model = MLP(input_dim=1000, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

__Training__

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for X, y in loader:
        optimizer.zero_grad()
        probs = model(X)
        loss = criterion(probs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += ((probs > 0.5) == y.bool()).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    for X, y in loader:
        correct += ((model(X) > 0.5) == y.bool()).sum().item()
    return correct / len(loader.dataset)


for epoch in range(1, 11):
    loss, train_acc = train_epoch(model, train_loader)
    print(f"Epoch {epoch:2d} | Loss: {loss:.4f} | Train accuracy: {train_acc:.3f}")

__Evaluation__

In [ ]:
# Benchmark: logistic regression
lr = LogisticRegression(C=np.inf, max_iter=1000)  # C=np.inf disables regularisation
lr.fit(X_train, y_train.ravel())
y_hat_lr = lr.predict(X_test).reshape(-1, 1)

# Neural network
test_accuracy = evaluate(model, test_loader)

print(f"Logistic regression test accuracy: {(y_hat_lr == y_test).mean():.3f}")
print(f"Neural network test accuracy:      {test_accuracy:.3f}")